## Cleaning Up the DataSet

In [1]:
# pandas + numpy are the only tools a cleaning stage needs: loading, validating,
# recoding and reshaping. Keeping this notebook dependency-light matters because every
# later notebook (and the README pipeline) depends on the CSV written at the end of it.

import pandas as pd
import numpy as np

In [2]:
# Why: before transforming anything, measure the damage. Row/column counts, duplicate
# rows and missing values decide WHICH cleaning steps are actually required (and whether
# any are optional). This cell only READS the file -- nothing is modified -- so the
# numbers describe the raw data exactly as delivered by Mendeley.

print("BOLLYWOOD MOVIE DATASET")
print("=" * 40)

df = pd.read_csv('DataSet.csv')

print(f"Total movies: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Missing values: {df.isna().sum().sum():,}")

print("\nColumn names:")
print(df.columns.tolist())

BOLLYWOOD MOVIE DATASET
Total movies: 1,698
Total columns: 14
Duplicate rows: 0
Missing values: 0

Column names:
['Movie Name', 'Release Period', 'Whether Remake', 'Whether Franchise', 'Genre', 'New Actor', 'New Director', 'New Music Director', 'Lead Star', 'Director', 'Music Director', 'Number of Screens', 'Revenue(INR)', 'Budget(INR)']


In [3]:
# Why: a per-column audit (dtype, missing count/%, cardinality). It tells us which
# columns need recoding (text Yes/No flags), which will be dropped (free-text names we
# cannot honestly quantify), and whether any column has gaps that would break the numeric
# conversions later. Doing this BEFORE encoding prevents silent type surprises.

data_quality = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

display(data_quality.sort_values("Missing Values", ascending=False))

,Data Type,Missing Values,Missing (%),Unique Values
Movie Name,str,0,0.0,1698
Release Period,str,0,0.0,2
Whether Remake,str,0,0.0,2
Whether Franchise,str,0,0.0,2
Genre,str,0,0.0,14
New Actor,str,0,0.0,2
New Director,str,0,0.0,2
New Music Director,str,0,0.0,2
Lead Star,str,0,0.0,761
Director,str,0,0.0,1045


In [4]:
# Why drop_duplicates: a duplicated film would be double-counted in every group
# comparison and would inflate every test statistic. The re-read from CSV is technically
# redundant (df was never modified above) but it makes the cell self-contained and
# idempotent: re-running it always starts from the pristine raw file, so accidental
# cell-order changes cannot corrupt the data.

df = pd.read_csv('DataSet.csv')
df = df.drop_duplicates()
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Movie Name          1698 non-null   str  
 1   Release Period      1698 non-null   str  
 2   Whether Remake      1698 non-null   str  
 3   Whether Franchise   1698 non-null   str  
 4   Genre               1698 non-null   str  
 5   New Actor           1698 non-null   str  
 6   New Director        1698 non-null   str  
 7   New Music Director  1698 non-null   str  
 8   Lead Star           1698 non-null   str  
 9   Director            1698 non-null   str  
 10  Music Director      1698 non-null   str  
 11  Number of Screens   1698 non-null   int64
 12  Revenue(INR)        1698 non-null   int64
 13  Budget(INR)         1698 non-null   int64
dtypes: int64(3), str(11)
memory usage: 185.8 KB
None


#### Separating and cleaning each header of the new Cleaned Dataset

In [5]:
# Why the 'deviants' check BEFORE encoding: .map() silently turns any value not in
# the dict into NaN, which would only surface much later as a modelling error. Proving
# the column contains ONLY 'Normal'/'Holiday' first guarantees the 1/0 encoding below is
# lossless. Cells 7-8 repeat this same guard for the other flags -- verbose, but it
# documents the data contract explicitly.

# Finding if rows deviate from standard 
deviants = df[~df['Release Period'].isin(['Normal', 'Holiday'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

holiday_release = df["Release Period"].map({'Holiday': 1, 'Normal': 0}).to_frame(name="Holiday_Release").astype(bool)
print(holiday_release.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Holiday_Release  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None


In [6]:
# Why: same pattern as cell 6, for the remake flag -- validate the domain {'Yes','No'}
# before replacing, so the 1/0 encoding cannot silently create NaNs from a typo'd label
# in the source data.

# Finding if rows deviate from standard Yes / No
deviants = df[~df['Whether Remake'].isin(['Yes', 'No'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

is_remake = df[["Whether Remake"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(is_remake.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Whether Remake  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None


In [7]:
# Why: identical guard for the franchise flag. Three near-identical cells on purpose:
# each flag is validated independently so a bad value in one column is attributed to the
# right source column in any future data refresh.

# Finding if rows deviate from standard Yes / No
deviants = df[~df['Whether Franchise'].isin(['Yes', 'No'])]
print("Deviants : ", len(deviants))    #0 : No deviations.

is_Franchise = df[["Whether Franchise"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(is_Franchise.info())

Deviants :  0
<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   Whether Franchise  1698 non-null   bool 
dtypes: bool(1)
memory usage: 1.8 KB
None


In [8]:
# Why: the three 'New *' flags are the dataset's own proxy for crew novelty, which the
# regression (notebook 3) needs as predictors. The written note records a scoping
# decision: Lead Star / Director names are deliberately NOT carried into the cleaned
# file because popularity cannot be encoded honestly into a number; notebook 2 re-joins
# them from the raw file for descriptive counts only.

newcrew = df[["New Actor", "New Director", "New Music Director"]].replace({'Yes': 1, 'No': 0}).astype(bool)
print(newcrew.info())

# We will be ignoring the names of actors and directors as it is extremetly difficult to encode their relative popularity
# accurately into variables, hence we arre only uses these 3 columns already provided in the dataset 
# and ignoring the next three columns.

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   New Actor           1698 non-null   bool 
 1   New Director        1698 non-null   bool 
 2   New Music Director  1698 non-null   bool 
dtypes: bool(3)
memory usage: 5.1 KB
None


In [9]:
# Why: class-balance audit before modelling. A flag split 95/5 (e.g. franchise) has
# very little statistical power, so knowing counts up front sets realistic expectations
# for which hypotheses can reach significance. It also re-verifies that every flag is now
# strictly 0/1 after encoding.

category_columns = [
    "Release Period",
    "Whether Remake",
    "Whether Franchise",
    "New Actor",
    "New Director",
    "New Music Director"
]

for column in category_columns:
    if column in df.columns:
        print(f"\n{'=' * 45}")
        print(column)
        print("=" * 45)

        counts = df[column].value_counts(dropna=False)
        percentages = (
            df[column].value_counts(normalize=True, dropna=False) * 100
        )

        summary = pd.DataFrame({
            "Movie Count": counts,
            "Percentage (%)": percentages.round(2)
        })

        display(summary)


Release Period


,Movie Count,Percentage (%)
Release Period,,
Normal,1052,61.96
Holiday,646,38.04



Whether Remake


,Movie Count,Percentage (%)
Whether Remake,,
No,1588,93.52
Yes,110,6.48



Whether Franchise


,Movie Count,Percentage (%)
Whether Franchise,,
No,1618,95.29
Yes,80,4.71


... (2 outputs left out)


New Director


,Movie Count,Percentage (%)
New Director,,
No,909,53.53
Yes,789,46.47



New Music Director


,Movie Count,Percentage (%)
New Music Director,,
No,1194,70.32
Yes,504,29.68


In [10]:
# Why one binary column per genre: genre is nominal, so coding it 1..14 would impose a
# fake ordering on it. drop_first=True avoids the dummy-variable trap: with all 14 dummies
# plus an intercept the design matrix would be perfectly collinear and OLS could not be
# solved. This is exactly why the regression in notebook 3 can invert normally.

# Getting all genre cause we need separate binary variable for each since we cannot encode their discrete values using a single variable.   
# However, we will be skippng one of the genre to avoid thedummy variable trap, we need all the variables to be 
# linearly independent for regression to function properly.

genres = pd.get_dummies(df[["Genre"]], columns=["Genre"], drop_first=True, dtype=bool)
print(genres.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Genre_adult         1698 non-null   bool 
 1   Genre_animation     1698 non-null   bool 
 2   Genre_comedy        1698 non-null   bool 
 3   Genre_documentary   1698 non-null   bool 
 4   Genre_drama         1698 non-null   bool 
 5   Genre_fantasy       1698 non-null   bool 
 6   Genre_horror        1698 non-null   bool 
 7   Genre_love_story    1698 non-null   bool 
 8   Genre_masala        1698 non-null   bool 
 9   Genre_mythological  1698 non-null   bool 
 10  Genre_rom_com       1698 non-null   bool 
 11  Genre_suspense      1698 non-null   bool 
 12  Genre_thriller      1698 non-null   bool 
dtypes: bool(13)
memory usage: 21.7 KB
None


In [11]:
# Why: same balance audit as cell 10 but for the encoded genres. It documents which
# genre became the (dropped) baseline category and flags near-empty genres whose
# coefficients will be unstable in the regression.

genre_counts = (
    df["Genre"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .value_counts()
)

genre_summary = pd.DataFrame({
    "Movie Count": genre_counts,
    "Percentage (%)": (
        genre_counts / len(df) * 100
    ).round(2)
})

display(genre_summary)

,Movie Count,Percentage (%)
Genre,,
drama,642,37.81
comedy,282,16.61
thriller,216,12.72
love_story,131,7.71
action,126,7.42
rom_com,97,5.71
adult,76,4.48
horror,54,3.18
suspense,29,1.71


In [12]:
# Why this composite: the whole project hinges on one 'scale' control, and budget and
# screens are two facets of the same money. log1p tames the right-skew so blockbusters do
# not dominate the min-max normalisation; the 0.5/0.5 weights are an explicit, documented
# assumption (equal trust in both facets); stretching to 1-10 makes the 10-bin
# stratification used everywhere downstream readable. It is a heuristic index built ONCE,
# here, so every notebook controls for the same definition.

# We are using a normalized log scale for budget and screens to create a composite score that is then mapped to a 1-10 scale.

w_budget = 0.5  # Budget Weight : Assumed to be equal to screens weight, but will be adjusted later based on futher data analysis.
w_screens = 0.5 # Screens Weight

log_budget = np.log1p(df["Budget(INR)"])
log_screens = np.log1p(df["Number of Screens"])


norm_budget = (log_budget - log_budget.min()) / (log_budget.max() - log_budget.min())
norm_screens = (log_screens - log_screens.min()) / (log_screens.max() - log_screens.min())

composite = (w_budget * norm_budget) + (w_screens * norm_screens)

movie_scale = (1 + 9 * composite).round(3).to_frame(name="Movie_Scale")
print(movie_scale.info())


<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Movie_Scale  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [13]:
# Why: revenue per screen is the scale-neutral revenue measure -- a 3,000-screen
# release needs far more gross than a 100-screen one just to break even. Deriving it here
# means downstream notebooks never re-divide raw columns (and risk divide inconsistencies)
# themselves.

rps = (df["Revenue(INR)"] / df["Number of Screens"]).round(1).to_frame(name="Revenue Per Screen")
print(rps.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Revenue Per Screen  1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [14]:
# Why: ROI (revenue/budget) is the project's outcome variable -- the ratio removes the
# mechanical advantage of a big budget, so 'did the strategy pay' is measured relative to
# the money at risk. Two notes: the header comment says 'Return On Interest' but the
# quantity is Return On Investment (typo); and no log is taken yet because notebook 5
# shows the raw ratio is heavy-tailed and chooses winsorisation over logs.

# our final point of interest : the Return On Interest

roi = (df["Revenue(INR)"] / df["Budget(INR)"]).round(5).to_frame(name="ROI")
print(roi.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   ROI     1698 non-null   float64
dtypes: float64(1)
memory usage: 13.4 KB
None


In [15]:
# Why: assemble the analysis frame in exactly one place. Movie Name is kept only as
# the join key back to the raw file; the original text columns (Release Period, Whether
# Remake..., Genre) are dropped by omission since encoded versions exist; monetary columns
# stay raw because the log transform is applied where it is needed (notebooks 3/5). One
# canonical output file guarantees every later notebook analyses identical data.

## Appending all the cleaned columns

df_clean = pd.concat([df[["Movie Name"]], is_remake, is_Franchise, holiday_release, genres, newcrew, df[["Budget(INR)", "Revenue(INR)", "Number of Screens"]], rps, movie_scale, roi], axis=1)
print(df_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Movie Name          1698 non-null   str    
 1   Whether Remake      1698 non-null   bool   
 2   Whether Franchise   1698 non-null   bool   
 3   Holiday_Release     1698 non-null   bool   
 4   Genre_adult         1698 non-null   bool   
 5   Genre_animation     1698 non-null   bool   
 6   Genre_comedy        1698 non-null   bool   
 7   Genre_documentary   1698 non-null   bool   
 8   Genre_drama         1698 non-null   bool   
 9   Genre_fantasy       1698 non-null   bool   
 10  Genre_horror        1698 non-null   bool   
 11  Genre_love_story    1698 non-null   bool   
 12  Genre_masala        1698 non-null   bool   
 13  Genre_mythological  1698 non-null   bool   
 14  Genre_rom_com       1698 non-null   bool   
 15  Genre_suspense      1698 non-null   bool   
 16  Genre_thriller   

In [16]:
# Why: a two-row eyeball check that pd.concat aligned columns correctly (no shifted
# or all-NaN rows) BEFORE the frame is written to disk. Catching misalignment here is
# cheap; catching it in notebook 3 would be expensive.

print(df_clean.head(2))

   Movie Name  Whether Remake  Whether Franchise  Holiday_Release  \
0  Qaidi Band           False              False             True   
1   Chaatwali           False              False             True   

   Genre_adult  Genre_animation  Genre_comedy  Genre_documentary  Genre_drama  \
0        False            False         False              False         True   
1         True            False         False              False        False   

   Genre_fantasy  ...  Genre_thriller  New Actor  New Director  \
0          False  ...           False       True         False   
1          False  ...           False       True          True   

   New Music Director  Budget(INR)  Revenue(INR)  Number of Screens  \
0               False    210000000       4500000                800   
1                True      1000000       1075000                  1   

   Revenue Per Screen  Movie_Scale      ROI  
0              5625.0        7.803  0.02143  
1           1075000.0        1.576  1.07500

In [17]:
# Why: the cleaned CSV is the contract between this notebook and everything else
# (the README pipeline order exists because of it). index=False because Movie Name is the
# key; a CSV rather than a pickle keeps it human-inspectable and diff-able in git.

## Writing back to a new CSV file for further analysis and model building.
df_clean.to_csv('CleanedDataSet.csv', index=False)